# Hands-on: Fine-tuning LLM dengan QLoRA

**Tujuan:** menjalankan alur fine-tuning end-to-end menggunakan QLoRA pada model kecil, sehingga seluruh proses dapat diselesaikan dalam beberapa menit di Google Colab (GPU T4, free tier).

**Yang akan dipelajari:**
1. Memuat model dalam presisi 4-bit (`BitsAndBytesConfig`)
2. Memasang adapter LoRA (`LoraConfig`) dan memverifikasi jumlah trainable parameters
3. Menyiapkan dataset instruksi berbahasa Indonesia
4. Menjalankan training dengan `SFTTrainer`
5. Membandingkan output model sebelum vs sesudah fine-tuning
6. Menyimpan dan memuat kembali adapter

> **Sebelum mulai:** pastikan runtime menggunakan GPU. Menu **Runtime → Change runtime type → T4 GPU**.

## 1. Instalasi Library

Stack standar untuk QLoRA:
- **transformers** — memuat model dan tokenizer
- **peft** — implementasi LoRA (Parameter-Efficient Fine-Tuning)
- **bitsandbytes** — kuantisasi 4-bit
- **trl** — `SFTTrainer` untuk supervised fine-tuning
- **datasets** — pengelolaan dataset

Instalasi memakan waktu 1-2 menit.

In [ ]:
%pip install -q -U transformers peft bitsandbytes trl datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.1 MB/s eta 0:00:00


In [ ]:
import torch

# Verifikasi GPU tersedia
assert torch.cuda.is_available(), "GPU tidak terdeteksi. Ubah runtime ke T4 GPU."
print(f"GPU        : {torch.cuda.get_device_name(0)}")
print(f"VRAM total : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU        : Tesla T4
VRAM total : 15.6 GB


## 2. Memuat Model dalam 4-bit

Inti dari **QLoRA**: base model dimuat dalam presisi 4-bit sehingga jejak memorinya turun sekitar 4x dibanding FP16.

Konfigurasi penting pada `BitsAndBytesConfig`:

| Parameter | Nilai | Penjelasan |
|---|---|---|
| `load_in_4bit` | `True` | Bobot base model disimpan dalam 4-bit |
| `bnb_4bit_quant_type` | `"nf4"` | NormalFloat4 — tipe data optimal untuk bobot berdistribusi normal |
| `bnb_4bit_use_double_quant` | `True` | Konstanta kuantisasi ikut dikuantisasi (hemat ±0,4 bit/parameter) |
| `bnb_4bit_compute_dtype` | `bfloat16`/`float16` | Presisi saat komputasi (dequantize on-the-fly) |

Model yang digunakan: **Qwen2.5-0.5B-Instruct** — cukup kecil agar training cepat, namun sudah instruction-tuned sehingga responsnya bisa dibandingkan sebelum/sesudah.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# T4 tidak mendukung bfloat16 secara native -> gunakan float16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Memori model 4-bit: {model.get_memory_footprint() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Memori model 4-bit: 0.45 GB


## 3. Baseline: Output Model Sebelum Fine-tuning

Sebelum training, kita rekam dulu bagaimana model menjawab beberapa prompt uji. Ini menjadi **baseline** pembanding.

Skenario yang digunakan: asisten layanan pelanggan sebuah perusahaan gadai fiktif bernama **GadaiKita**, dengan format jawaban yang harus konsisten (salam pembuka, jawaban singkat, penutup baku). Format seperti ini sulit dijamin hanya lewat prompting — inilah kasus yang cocok untuk fine-tuning.

In [ ]:
def generate(model, prompt, max_new_tokens=150):
    """Membangun chat prompt, menjalankan inference, dan mengembalikan teks jawaban."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )

PROMPT_UJI = [
    "Apa syarat mengajukan gadai emas?",
    "Berapa lama proses pencairan dana gadai?",
    "Apakah barang saya aman selama digadaikan?",
]

print("=== OUTPUT SEBELUM FINE-TUNING ===")
for p in PROMPT_UJI:
    print(f"\n[Prompt] {p}")
    print(f"[Jawaban] {generate(model, p)}")

=== OUTPUT SEBELUM FINE-TUNING ===

[Prompt] Apa syarat mengajukan gadai emas?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[Jawaban] Tentang pengajian emas di Indonesia, ada beberapa syarat yang harus diterima:

1. Pendidikan: Anda perlu memiliki pendidikan universitas atau kualifikasi profesional yang mencakupkan pengetahuan dan kemampuan untuk memilih emas sebagai sumber keamanan.

2. Keterampilan: Anda perlu memiliki kemampuan dalam analisis data, pemahaman lingkungan, dan penilaian perilaku.

3. Keberanian: Anda perlu memiliki keberanian untuk berinteraksi dengan orang lain dan melakukan tindakan positif.

4. Keberkesanan: Anda perlu memiliki keberkesanan terhadap keragaman dan kes

[Prompt] Berapa lama proses pencairan dana gadai?
[Jawaban] Proses pencairan dana di Indonesia adalah proses yang cukup lambat dan berlangsung selama beberapa tahun. Namun, ada beberapa faktor yang dapat mempengaruhi waktu pencairan:

1. Kebutuhan dan kebutuhannya sendiri: Setiap individu memiliki kebutuhan dan kemampuan untuk mengelola dana mereka dengan cepat.

2. Penyusunan dan pengurusan: Dalam beberapa negara, pencaira

**Amati:** jawaban model bersifat generik — tidak menyebut nama perusahaan, tidak memakai format salam/penutup tertentu, dan panjangnya tidak terkontrol. Target fine-tuning kita: jawaban selalu mengikuti pola

```
Halo, terima kasih telah menghubungi GadaiKita.
<jawaban singkat 1-2 kalimat>
Ada lagi yang bisa kami bantu?
```

## 4. Menyiapkan Dataset Instruksi

Dataset dibangun langsung di notebook (untuk keperluan demo) dalam **format messages** (chat), lalu dikonversi menjadi objek `Dataset`.

Prinsip penting dalam praktik nyata:
- **Kualitas mengalahkan kuantitas** — untuk adaptasi gaya/format, ratusan contoh berkualitas sering lebih efektif daripada ribuan contoh asal-asalan.
- Contoh harus **konsisten** dengan perilaku yang diinginkan, karena model akan meniru pola yang paling sering muncul.
- Untuk demo ini, kita membuat pasangan tanya-jawab dari template agar volume cukup untuk terlihat efeknya dalam training singkat.

In [ ]:
from datasets import Dataset
import random

random.seed(42)

SALAM = "Halo, terima kasih telah menghubungi GadaiKita."
PENUTUP = "Ada lagi yang bisa kami bantu?"

def jawaban(inti):
    return f"{SALAM} {inti} {PENUTUP}"

# Basis pengetahuan tanya-jawab (skenario layanan pelanggan GadaiKita)
qa_pairs = [
    ("Apa syarat mengajukan gadai emas?",
     "Syaratnya cukup KTP asli dan barang emas yang akan digadaikan; proses dapat dilakukan di seluruh cabang GadaiKita."),
    ("Berapa lama proses pencairan dana gadai?",
     "Proses pencairan dana di GadaiKita rata-rata hanya 15 menit setelah taksiran barang disetujui."),
    ("Apakah barang saya aman selama digadaikan?",
     "Barang Anda disimpan di ruang khusus berstandar keamanan tinggi dan diasuransikan penuh selama masa gadai."),
    ("Bagaimana cara memperpanjang masa gadai?",
     "Perpanjangan dapat dilakukan dengan membayar biaya pemeliharaan di cabang atau melalui aplikasi GadaiKita sebelum jatuh tempo."),
    ("Berapa bunga atau biaya gadai per bulan?",
     "Biaya pemeliharaan GadaiKita mulai dari 1 persen per 15 hari, tergantung golongan pinjaman."),
    ("Apakah bisa menebus barang sebelum jatuh tempo?",
     "Tentu bisa; Anda dapat menebus barang kapan saja dan biaya hanya dihitung sesuai masa pakai."),
    ("Barang apa saja yang bisa digadaikan?",
     "GadaiKita menerima emas, perhiasan, kendaraan bermotor, serta barang elektronik tertentu."),
    ("Bagaimana jika saya terlambat membayar?",
     "Ada masa tenggang; jika melewati batas, barang akan dilelang dan kelebihan hasil lelang dikembalikan kepada Anda."),
    ("Apakah ada aplikasi untuk cek status gadai?",
     "Ada; unduh aplikasi GadaiKita untuk memantau status gadai, jatuh tempo, dan pembayaran secara online."),
    ("Bisakah gadai diwakilkan orang lain?",
     "Pengajuan gadai harus dilakukan pemilik barang, namun pembayaran cicilan dapat diwakilkan."),
]

# Variasi cara bertanya agar model belajar pola, bukan menghafal kalimat
VARIASI = [
    "{q}", "{q} Mohon infonya.", "Mau tanya, {q_lower}",
    "Permisi, {q_lower}", "{q} Terima kasih.", "Halo kak, {q_lower}",
]

rows = []
for q, a in qa_pairs:
    for v in VARIASI:
        pertanyaan = v.format(q=q, q_lower=q[0].lower() + q[1:])
        rows.append({
            "messages": [
                {"role": "user", "content": pertanyaan},
                {"role": "assistant", "content": jawaban(a)},
            ]
        })

random.shuffle(rows)
dataset = Dataset.from_list(rows)
print(f"Jumlah contoh: {len(dataset)}")
print("\nContoh data:")
print(dataset[0]["messages"][0]["content"])
print(dataset[0]["messages"][1]["content"])

Jumlah contoh: 60

Contoh data:
Mau tanya, barang apa saja yang bisa digadaikan?
Halo, terima kasih telah menghubungi GadaiKita. GadaiKita menerima emas, perhiasan, kendaraan bermotor, serta barang elektronik tertentu. Ada lagi yang bisa kami bantu?


## 5. Konfigurasi LoRA

Di sinilah konsep dari materi diterapkan. `LoraConfig` menentukan bagaimana adapter dipasang:

| Parameter | Nilai | Penjelasan |
|---|---|---|
| `r` | 16 | Rank matriks adapter — dimensi "bottleneck" pada dekomposisi ΔW = B·A |
| `lora_alpha` | 32 | Faktor skala; kontribusi adapter sebesar `alpha/r` (konvensi umum: alpha = 2r) |
| `target_modules` | semua linear layer | Adapter dipasang di proyeksi attention dan MLP |
| `lora_dropout` | 0.05 | Regularisasi ringan untuk dataset kecil |
| `task_type` | `CAUSAL_LM` | Jenis task: language modeling kausal |

Setelah dipasang, perhatikan output `print_trainable_parameters()` — **hanya sebagian kecil parameter yang dilatih**, sementara base model 4-bit tetap frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Persiapan standar untuk training di atas model terkuantisasi
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 6. Training dengan SFTTrainer

`SFTTrainer` dari library **TRL** menangani detail teknis: menerapkan chat template pada kolom `messages`, tokenisasi, dan loop training.

Beberapa keputusan konfigurasi:
- `num_train_epochs=3` dan dataset kecil → training selesai dalam hitungan menit.
- `learning_rate=2e-4` — nilai umum untuk LoRA (lebih tinggi dari full fine-tuning karena parameter yang dilatih sedikit).
- `paged_adamw_8bit` — optimizer hemat memori, komponen ketiga dari QLoRA.
- `gradient_accumulation_steps=4` — batch efektif 16 tanpa menambah kebutuhan VRAM.

**Yang perlu diamati saat training berjalan:** kolom `Loss` — turun secara konsisten menandakan model mempelajari pola dataset.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="qlora-gadaikita",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=2,
    optim="paged_adamw_8bit",
    max_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
2,3.730129
4,2.589508
6,1.756149
8,1.335075


## 7. Evaluasi: Sebelum vs Sesudah

Jalankan kembali prompt uji yang sama. Bandingkan dengan output pada Bagian 3:
- Apakah jawaban kini selalu diawali salam GadaiKita dan diakhiri kalimat penutup baku?
- Apakah isi jawaban sesuai basis pengetahuan pada dataset?
- Coba juga **pertanyaan yang tidak ada di dataset** — apakah polanya tetap terbawa (generalisasi format)?

Catatan penting untuk kelas: **loss yang turun bukan bukti akhir** — evaluasi perilaku pada prompt nyata seperti inilah yang menentukan apakah fine-tuning berhasil.

In [ ]:
model.eval()

print("=== OUTPUT SESUDAH FINE-TUNING ===")
for p in PROMPT_UJI:
    print(f"\n[Prompt] {p}")
    print(f"[Jawaban] {generate(model, p)}")

# Pertanyaan di luar dataset -> menguji generalisasi format
print("\n[Prompt - di luar dataset] Apakah kantor buka hari Minggu?")
print(f"[Jawaban] {generate(model, 'Apakah kantor buka hari Minggu?')}")

## 8. Menyimpan dan Memuat Kembali Adapter

Keunggulan operasional LoRA: yang disimpan **hanya adapter**, bukan seluruh model. Ukurannya puluhan MB, sehingga:
- Mudah dibagikan dan di-versioning
- Beberapa adapter (per use case) dapat berbagi satu base model yang sama
- Untuk deployment, adapter dapat di-**merge** ke base model menjadi satu file (`merge_and_unload()`)

In [ ]:
ADAPTER_DIR = "adapter-gadaikita"
trainer.save_model(ADAPTER_DIR)

import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Ukuran adapter: {total / 1e6:.1f} MB")
print("Bandingkan dengan base model 0.5B (sekitar 1 GB dalam FP16).")

In [ ]:
# Simulasi deployment: muat base model baru + pasang adapter dari disk
from peft import PeftModel

base_reloaded = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model_reloaded = PeftModel.from_pretrained(base_reloaded, ADAPTER_DIR)
model_reloaded.eval()

print("[Uji model hasil load ulang]")
print(generate(model_reloaded, "Apa syarat mengajukan gadai emas?"))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[Uji model hasil load ulang]
Halo, terima kasih telah menghubungi GadaiEmas. Syarat pengajuan gadai emas termasuk memenuhi kewajiban pajak dan memenuhi standar kualitas gadai emas yang diatur oleh Badan Pusat Penyelidikan Emas (BPE). Ada lagi yang bisa kami bantu?


## 9. Rangkuman dan Diskusi

**Yang sudah dilakukan:**
1. Memuat model dalam 4-bit (NF4 + double quantization) — komponen kuantisasi QLoRA
2. Memasang adapter LoRA dengan `r=16` — hanya sekitar 1% parameter yang dilatih
3. Fine-tuning format dan persona layanan pelanggan dengan dataset kecil
4. Membuktikan perubahan perilaku lewat perbandingan sebelum/sesudah
5. Menyimpan adapter berukuran puluhan MB yang dapat di-load ulang

**Bahan diskusi kelas:**
- Pada kasus GadaiKita di atas, informasi mana yang lebih tepat ditangani **RAG** (misal: harga emas hari ini) dan mana yang tepat lewat **fine-tuning** (format dan persona)?
- Apa risiko jika basis pengetahuan dimasukkan ke bobot model lewat fine-tuning, lalu kebijakan perusahaan berubah?
- Bagaimana strategi evaluasi yang lebih sistematis dibanding pemeriksaan manual? (test set terpisah, LLM-as-a-judge, metrik format compliance)

**Eksperimen lanjutan yang disarankan:**
- Ubah `r` menjadi 4 dan 64 — amati perubahan jumlah trainable parameters dan kualitas hasil
- Kurangi dataset menjadi 10 contoh — amati kapan model mulai gagal konsisten
- Ganti model dengan yang lebih besar (misal 1.5B) — bandingkan kebutuhan VRAM